In [60]:
import os
from dotenv import load_dotenv
from langchain_openrouter import ChatOpenRouter

load_dotenv(dotenv_path=".env", override=True)

api_key = os.getenv("OPENROUTER_API_KEY")
if not api_key:
    raise RuntimeError("OPENROUTER_API_KEY is missing from .env")

llm = ChatOpenRouter(
    model="deepseek/deepseek-v4-flash-0731",
    api_key=api_key,
    temperature=0,
    max_tokens=500,
)

In [ ]:
# response = llm.invoke("What is AI? Max 300 words.")
# print(response.content)

AI, or Artificial Intelligence, is the field of computer science focused on creating systems that can perform tasks typically requiring human intelligence. These tasks include learning, reasoning, problem-solving, perception, language understanding, and decision-making.

At its core, AI works by using algorithms and large amounts of data to identify patterns and make predictions or decisions. There are two main types:

- **Narrow AI** (or Weak AI) is designed for a specific task, such as voice assistants, recommendation systems, or facial recognition. Most AI today is narrow.
- **General AI** (or Strong AI) would possess human-like cognitive abilities across a wide range of tasks. This remains theoretical and does not yet exist.

A key subset of AI is **machine learning**, where systems improve automatically through experience. Deep learning, a further subset, uses neural networks with many layers to process complex data like images and speech.

AI is already embedded in daily life: se

## **RAG IMPLEMENTATION with own TEXT data**

#### **Step 1: Preparing Document for Text**

In [34]:
from langchain_core.documents import Document

#Your text
my_text = """Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics, and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximise their chances of achieving defined goals.[1]

High-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, play and analysis in strategy games (e.g., chess and Go), and content generation (e.g. images, audio, and videos).

The traditional goals of AI research include learning, reasoning, knowledge representation, planning, natural language processing, and perception, as well as support for robotics.[a] To reach these goals, AI researchers use techniques including state space search and mathematical optimisation, formal logic, artificial neural networks, and methods based on statistics, operations research, and economics.[b] AI also draws upon psychology, linguistics, philosophy, neuroscience, and other fields.[2] Some companies, such as OpenAI, Google DeepMind, and Meta, aim to create artificial general intelligence (AGI)—AI that can complete nearly any cognitive task at least as well as a human.[3]

Artificial intelligence was founded as an academic discipline in 1956.[4] The field went through multiple cycles of optimism throughout its history,[5][6] followed by periods of disappointment and loss of funding, known as AI winters.[7][8] Funding and interest increased substantially after 2012, when graphics processing units (GPUs) started being used to accelerate neural networks, and deep learning outperformed previous AI techniques.[9] This growth accelerated further after 2017 with the transformer architecture.[10] In the 2020s, an AI boom coincided with advances in generative AI, which became widespread and allowed for the creation and modification of media. In addition to AI safety and unintended consequences and harms from the use of AI, ethical concerns, AI's long-term effects, environmental effects,[11] and potential existential risks have prompted discussions of AI regulation.
"""

#use langchain to create document page content with your text
docs = [Document(page_content=my_text, metadata={"source":"Wikipedia","documentID":"Doc1"})]



#### **Step 2: CHUNKING**

In [35]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = splitter.split_documents(docs)
chunks

[Document(metadata={'source': 'Wikipedia', 'documentID': 'Doc1'}, page_content='Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics, and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximise their chances of achieving defined goals.[1]'),
 Document(metadata={'source': 'Wikipedia', 'documentID': 'Doc1'}, page_content='High-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, play and analysis in strategy games (e.g., chess and Go), and content generation (e.g. images, audio, and videos).'),
 Document(metadata={'source': 'Wikipedia', 'documentID': 'Doc1'}, page_content='The traditional goals 

#### **STEP 3: Creating EMBEDDINGS**

In [ ]:
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(
    model="qwen/qwen3-embedding-8b",
    api_key=api_key,
    base_url="https://openrouter.ai/api/v1",
    # IMPORTANT for OpenRouter
    check_embedding_ctx_length=False,
    # Recommended for OpenRouter-compatible providers
    # encoding_format="float",
)

#### **Step 4: Create and store embeddings in Vector store**

In [47]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model
)

#----------behind the scenes-----------
# vectorstore=[]
# for doc in chunks:
#   vector = embedding_model.embed_documents([doc.page_content])
#   vectorstore.append(vector)

#### **Step 5: SEMANTIC SEARCH** 

In [58]:
context = vectorstore.similarity_search("What can be benefits from AI in research?",k=4)
print(context)

[Document(metadata={'documentID': 'Doc1', 'source': 'Wikipedia'}, page_content='The traditional goals of AI research include learning, reasoning, knowledge representation, planning, natural language processing, and perception, as well as support for robotics.[a] To reach these goals, AI researchers use techniques including state space search and mathematical optimisation, formal logic, artificial neural networks, and methods based on statistics, operations research, and economics.[b] AI also draws upon psychology, linguistics, philosophy, neuroscience, and other fields.[2]'), Document(metadata={'documentID': 'Doc1', 'source': 'Wikipedia'}, page_content='The traditional goals of AI research include learning, reasoning, knowledge representation, planning, natural language processing, and perception, as well as support for robotics.[a] To reach these goals, AI researchers use techniques including state space search and mathematical optimisation, formal logic, artificial neural networks, a

#### **LETS TALK TO LLM FINALLY**

In [59]:
response_context = llm.invoke(f"How is AI being used in research fields?? You can answer using the following context: {context}")
print(response_context.content)

Based on the provided context, AI is used in research fields by applying techniques such as:

- **State space search and mathematical optimisation**
- **Formal logic**
- **Artificial neural networks**
- **Methods based on statistics, operations research, and economics**

These techniques help AI address core research goals like:

- **Learning**
- **Reasoning**
- **Knowledge representation**
- **Planning**
- **Natural language processing**
- **Perception**
- **Robotics**

AI research also draws on knowledge from fields like **psychology, linguistics, philosophy, neuroscience**, and others, making it highly interdisciplinary and useful across many research domains.
